In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

# ==============================
# 設定
# ==============================
START_URL = "https://www.musashino-u.ac.jp/"
DOMAIN = urlparse(START_URL).netloc  # 例: "www.musashino-u.ac.jp"

# 訪問済みURLを管理する集合
visited = set()

# サイトマップを保存する辞書
# key: URL, value: <title> の文字列
sitemap = {}


# ==============================
# HTML を取得する関数
# ==============================
def fetch_html(url):
    """
    指定した URL から HTML を取得する関数
    """
    headers = {
        # よくある User-Agent の設定（ブラウザっぽく見せる）
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    try:
        res = requests.get(url, headers=headers, timeout=10)
        res.raise_for_status()  # ステータスコードが 200 系以外なら例外
        res.encoding = res.apparent_encoding  # エンコーディング自動判定（sample と同じ考え方）
        return res.text
    except requests.RequestException as e:
        print(f"[ERROR] {url} の取得に失敗しました: {e}")
        return None


# ==============================
# title と 同一ドメインのリンクを抽出する関数
# ==============================
def parse_page(html, base_url):
    """
    HTML から <title> と 同一ドメインのリンク一覧を取得する
    """
    soup = BeautifulSoup(html, "html.parser")

    # ---- <title> の取得 ----
    title_tag = soup.title
    if title_tag is not None:
        title_text = title_tag.get_text(strip=True)
    else:
        title_text = ""

    # ---- a タグからリンクを取得 ----
    links = set()

    # コメントアウトされているリンクは BeautifulSoup の時点で基本的に取れないので OK
    # ただし href がない・特殊スキームのものは除外する
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()

        # 空文字、ページ内リンク(#始まり)、メールリンク、電話リンク、javascript はスキップ
        if (
            href == ""
            or href.startswith("#")
            or href.startswith("mailto:")
            or href.startswith("tel:")
            or href.startswith("javascript:")
        ):
            continue

        # 相対パス → 絶対URLに変換
        full_url = urljoin(base_url, href)

        # アンカー(#以下)は削除（同じページ扱い）
        full_url = full_url.split("#")[0]

        parsed = urlparse(full_url)

        # ドメインが違うものは除外（同一ドメインのみ）
        if parsed.netloc != DOMAIN:
            continue

        links.add(full_url)

    return title_text, links


# ==============================
# 幅優先 or 深さ優先でクロールする関数
# ==============================
def crawl_site(start_url, max_pages=200):
    """
    武蔵野大学Webサイトのサイトマップを取得するためのクローラ（骨組み）

    max_pages は安全のための上限（必要に応じて増やす or 外す）
    """
    to_visit = [start_url]

    while to_visit:
        url = to_visit.pop(0)  # queue（幅優先）。深さ優先なら pop() でもOK

        if url in visited:
            continue

        print(f"[FETCH] {url}")
        html = fetch_html(url)

        # 訪問済みに登録
        visited.add(url)

        if html is None:
            # 取得失敗したページはスキップ
            continue

        # ページ解析：title & 同一ドメインリンク
        title_text, links = parse_page(html, url)

        # サイトマップに登録
        sitemap[url] = title_text

        # 未訪問のリンクをキューに追加
        for link in links:
            if link not in visited and link not in to_visit:
                to_visit.append(link)

        # サーバへの負荷軽減
        time.sleep(1)  # 必ず sleep を入れる（秒数は必要に応じて変更）

        # ページ数の上限チェック（テスト時に便利）
        if len(visited) >= max_pages:
            print(f"[INFO] max_pages = {max_pages} に達したため終了します。")
            break


# ==============================
# 実行
# ==============================
crawl_site(START_URL, max_pages=200)  # 提出前に max_pages を増やす or 外す


# ==============================
# 結果の出力
# ==============================
for url, title in sitemap.items():
    print(url, " : ", title)


[FETCH] https://www.musashino-u.ac.jp/
[FETCH] https://www.musashino-u.ac.jp/student-life/fees/
[FETCH] https://www.musashino-u.ac.jp/guide/profile/message.html
[FETCH] https://www.musashino-u.ac.jp/academics/advanced_course/
[FETCH] https://www.musashino-u.ac.jp/guide/profile/history.html
[FETCH] https://www.musashino-u.ac.jp/basic/learning_cycle.html
[FETCH] https://www.musashino-u.ac.jp/academics/Lifelong_learning/
[FETCH] https://www.musashino-u.ac.jp/research/kakenhi/
[FETCH] https://www.musashino-u.ac.jp/news/detail/20251105-7385.html
[FETCH] https://www.musashino-u.ac.jp/news/detail/20251030-7383.html
[FETCH] https://www.musashino-u.ac.jp/business.html
[FETCH] https://www.musashino-u.ac.jp/admission/
[FETCH] https://www.musashino-u.ac.jp/news/detail/20251029-7375.html
[FETCH] https://www.musashino-u.ac.jp/event/detail/20251112-7302.html
[FETCH] https://www.musashino-u.ac.jp/academics/basic/
[FETCH] https://www.musashino-u.ac.jp/student-life/tool/
[FETCH] https://www.musashino-u.